# Notebook 08 — NLP Combos : extraction depuis transcription YouTube

**Objectif :** à partir d'une vidéo de gameplay/combo Yu-Gi-Oh, extraire automatiquement
les cartes mentionnées et modéliser la séquence comme un **graphe orienté de combo**.

**Pipeline :**
1. `youtube_transcript_api` → transcription gratuite sans clé API
2. Matching sur les noms de cartes de la DB (longest-match first)
3. Segmentation temporelle → séquences de combo
4. Graphe orienté A→B→C + visualisation pyvis

**Vidéo test :** https://www.youtube.com/watch?v=MsZb-dJAGHo

## Cell 1 — Imports & config

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
import sqlite3, re, os, json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import networkx as nx
from pyvis.network import Network
import warnings
warnings.filterwarnings('ignore')

DB = os.path.join(os.getcwd(), '..', 'data', 'yugioh.db')
VIDEO_URL = 'https://www.youtube.com/watch?v=ct9CxAYU5ZM'
VIDEO_ID  = 'ct9CxAYU5ZM'
ARCHETYPE = 'Kewl Tune'   # None si vidéo générique

# Fenêtre de combo : cartes mentionnées dans les X secondes suivantes = même séquence
COMBO_WINDOW_SEC = 30

# Corrections ASR : YouTube auto-transcrit mal les noms de cartes YGO
ASR_CORRECTIONS = {
    r'\bCool Tune\b': 'Kewl Tune',
    r'\bCooltune\b':  'Kewl Tune',
}

# Nicknames par archetype (nom abrégé utilisé à l'oral → nom complet DB)
ARCHETYPE_NICKNAMES = {
    'Kewl Tune': {
        r'\bHarmonia\b':    'Fydraulis Harmonia',
        r'\bClip\b':        'Kewl Tune Clip',
        r'\bMix\b':         'Kewl Tune Mix',
        r'\bCue\b':         'Kewl Tune Cue',
        r'\bRotary\b':      'Kewl Tune Rotary',
        r'\bReco\b':        'Kewl Tune Reco',
        r'\bRemix\b':       'Kewl Tune Remix',
        r'\bB2B\b':         'Kewl Tune B2B',
        r'\bPlaylist\b':    'Kewl Tune Playlist',
        r'\bTrack Maker\b': 'Kewl Tune Track Maker',
    }
}

def normalize_transcript(text: str, archetype: str | None = None) -> str:
    """Corrige les erreurs ASR communes + expand les nicknames d'archetype."""
    for pattern, replacement in ASR_CORRECTIONS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    if archetype and archetype in ARCHETYPE_NICKNAMES:
        for pattern, replacement in ARCHETYPE_NICKNAMES[archetype].items():
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

print(f'Video ID  : {VIDEO_ID}')
print(f'Archetype : {ARCHETYPE}')
print(f'DB        : {DB}')

## Cell 2 — Charger les noms de cartes depuis la DB

In [38]:
conn = sqlite3.connect(DB)

cards_df = pd.read_sql_query(
    'SELECT name FROM cards ORDER BY LENGTH(name) DESC',
    conn
)
conn.close()

card_names = cards_df['name'].tolist()

# Filtre 1 : noms trop courts (≤ 3 chars) → trop de faux positifs
card_names_filtered = [c for c in card_names if len(c) > 3]

# Filtre 2 : blacklist de mots anglais courants qui matchent des noms de cartes
BLACKLIST = {
    'NEXT', 'Fine', 'Return', 'Question', 'Last Turn', 'Honest',
    'Typhoon', 'Recycle', 'Beginning', 'Attack', 'Draw', 'Hand',
    'Change', 'Double', 'Final', 'First', 'Full', 'Good', 'Hard',
    'Hope', 'Just', 'Life', 'Lost', 'Main', 'Mark', 'Mind', 'More',
    'Once', 'Over', 'Play', 'Plus', 'Post', 'Pure', 'Real', 'Rite',
    'Rush', 'Same', 'Star', 'Stop', 'Take', 'This', 'Time', 'True',
    'Type', 'Used', 'Well', 'Will', 'With', 'Zero', 'Prohibition', 'Contact', 'Surface', 'Storm', 'Reasoning',
'Gamble', 'Gravity', 'Genesis', 'Road', 'Warrior', 'Maiden',
'Material', 'Preparation', 'Collapse', 'Visas'
}
card_names_filtered = [c for c in card_names_filtered if c not in BLACKLIST]

print(f'{len(card_names_filtered)} cartes chargées après filtres')
print(f'Blacklist appliquée : {len(BLACKLIST)} termes exclus')

# Pré-compiler les patterns regex (insensible à la casse)
card_patterns = [
    (name, re.compile(r'\b' + re.escape(name) + r'\b', re.IGNORECASE))
    for name in card_names_filtered
]
print(f'{len(card_patterns)} patterns compilés')

13777 cartes chargées après filtres
Blacklist appliquée : 64 termes exclus
13777 patterns compilés


## Cell 3 — Récupérer la transcription YouTube

In [ ]:
def fetch_transcript(video_id: str) -> list[dict]:
    for lang in (['en'], ['fr'], None):
        try:
            api = YouTubeTranscriptApi()
            transcript = api.fetch(video_id, **({'languages': lang} if lang else {}))
            data = [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in transcript]
            label = lang[0].upper() if lang else '?'
            print(f'Transcription {label} récupérée ({len(data)} segments)')
            return data
        except Exception:
            pass
    print('Aucune transcription disponible.')
    return []


raw_transcript = fetch_transcript(VIDEO_ID)

# Normalisation : ASR corrections + nickname expansion (dépend de ARCHETYPE)
transcript = [
    {'text': normalize_transcript(seg['text'], ARCHETYPE),
     'start': seg['start'], 'duration': seg['duration']}
    for seg in raw_transcript
]

if transcript:
    print(f'\nDurée totale estimée : {transcript[-1]["start"] + transcript[-1]["duration"]:.0f}s')
    print(f'Normalisation ASR appliquée (archetype: {ARCHETYPE})')
    print('\nExtrait (premiers segments) :')
    for seg in transcript[:8]:
        print(f'  [{seg["start"]:6.1f}s] {seg["text"]}')
else:
    print('Aucune transcription disponible pour cette vidéo.')

## Cell 4 — Extraction des cartes mentionnées

In [40]:
def extract_card_mentions(transcript: list[dict], card_patterns: list) -> list[dict]:
    """
    Pour chaque segment de la transcription, cherche les noms de cartes.
    Retourne une liste de mentions : {card, start, text}
    Longest-match : une fois une carte trouvée dans le texte, on la retire pour éviter
    les sous-matches (ex: 'Snake-Eye' vs 'Snake-Eye Ash').
    """
    mentions = []
    for seg in transcript:
        text = seg['text']
        found_in_seg = set()
        remaining = text
        for card_name, pattern in card_patterns:   # déjà triés longest-first
            if pattern.search(remaining):
                found_in_seg.add(card_name)
                # Masquer pour éviter les sous-matches
                remaining = pattern.sub('', remaining)
        for card in found_in_seg:
            mentions.append({'card': card, 'start': seg['start'], 'text': seg['text']})
    return mentions


if transcript:
    mentions = extract_card_mentions(transcript, card_patterns)
    mentions_df = pd.DataFrame(mentions).sort_values('start').reset_index(drop=True)

    print(f'{len(mentions_df)} mentions de cartes détectées')
    print(f'{mentions_df["card"].nunique()} cartes uniques\n')

    # Top cartes les plus mentionnées
    top_cards = mentions_df['card'].value_counts().head(20)
    print('Top 20 cartes les plus mentionnées :')
    print(top_cards.to_string())
else:
    mentions_df = pd.DataFrame(columns=['card', 'start', 'text'])
    print('Pas de transcription — mentions vides.')

16 mentions de cartes détectées
9 cartes uniques

Top 20 cartes les plus mentionnées :
card
Forbidden Crown                      5
Branded Opening                      3
Synchro Material                     2
One for One                          1
Fallen of Albaz                      1
Warning Point                        1
Libromancer Prevented                1
Fallen of the White Dragon           1
The Dragon that Devours the Dogma    1


## Cell 5 — Segmentation en séquences de combo

In [41]:
def segment_combos(mentions_df: pd.DataFrame, window_sec: float = 30) -> list[list[str]]:
    """
    Regroupe les mentions en séquences de combo :
    si deux mentions sont à moins de `window_sec` secondes, elles font partie du même combo.
    Retourne une liste de séquences (listes de noms de cartes ordonnées dans le temps).
    """
    if mentions_df.empty:
        return []
    combos = []
    current = []
    last_time = -999
    for _, row in mentions_df.iterrows():
        if row['start'] - last_time > window_sec and current:
            combos.append(current)
            current = []
        current.append(row['card'])
        last_time = row['start']
    if current:
        combos.append(current)
    return combos


combos = segment_combos(mentions_df, window_sec=COMBO_WINDOW_SEC)

print(f'{len(combos)} séquences de combo détectées (fenêtre = {COMBO_WINDOW_SEC}s)\n')

# Afficher les plus longues séquences
combos_sorted = sorted(combos, key=len, reverse=True)
print('Top 5 séquences les plus longues :')
for i, combo in enumerate(combos_sorted[:5]):
    print(f'  Combo {i+1} ({len(combo)} cartes) : {" → ".join(combo[:8])}{"..." if len(combo)>8 else ""}')

8 séquences de combo détectées (fenêtre = 30s)

Top 5 séquences les plus longues :
  Combo 1 (4 cartes) : Forbidden Crown → Forbidden Crown → Warning Point → Libromancer Prevented
  Combo 2 (3 cartes) : Branded Opening → The Dragon that Devours the Dogma → Branded Opening
  Combo 3 (2 cartes) : Forbidden Crown → Synchro Material
  Combo 4 (2 cartes) : Synchro Material → Fallen of Albaz
  Combo 5 (2 cartes) : Forbidden Crown → Forbidden Crown


## Cell 6 — Graphe orienté de combos

In [42]:
def build_combo_graph(combos: list[list[str]]) -> nx.DiGraph:
    """
    Construit un graphe orienté où une arête A→B signifie
    que la carte B est jouée après la carte A dans au moins une séquence de combo.
    Le poids de l'arête = nombre de fois que la transition A→B apparaît.
    """
    G = nx.DiGraph()
    for combo in combos:
        for i in range(len(combo) - 1):
            a, b = combo[i], combo[i+1]
            if a == b:
                continue
            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)
    return G


G = build_combo_graph(combos)

print(f'Graphe de combo : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes')

# Top transitions les plus fréquentes
if G.number_of_edges() > 0:
    edges = sorted(G.edges(data=True), key=lambda e: e[2]['weight'], reverse=True)
    print('\nTop 15 transitions A→B :')
    for a, b, d in edges[:15]:
        print(f'  {a} → {b}  (×{d["weight"]})')

# Nœuds centraux (in-degree = cartes vers lesquelles on joue le plus)
if G.number_of_nodes() > 0:
    in_deg = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:5]
    out_deg = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:5]
    print('\nCartes "destination" (in-degree élevé) :', [n for n,_ in in_deg])
    print('Cartes "source"      (out-degree élevé) :', [n for n,_ in out_deg])

Graphe de combo : 7 nœuds, 6 arêtes

Top 15 transitions A→B :
  Forbidden Crown → Synchro Material  (×1)
  Forbidden Crown → Warning Point  (×1)
  Synchro Material → Fallen of Albaz  (×1)
  Warning Point → Libromancer Prevented  (×1)
  Branded Opening → The Dragon that Devours the Dogma  (×1)
  The Dragon that Devours the Dogma → Branded Opening  (×1)

Cartes "destination" (in-degree élevé) : ['Synchro Material', 'Fallen of Albaz', 'Warning Point', 'Libromancer Prevented', 'Branded Opening']
Cartes "source"      (out-degree élevé) : ['Forbidden Crown', 'Synchro Material', 'Warning Point', 'Branded Opening', 'The Dragon that Devours the Dogma']


## Cell 7 — Visualisation pyvis du graphe

In [43]:
def visualize_combo_graph(G: nx.DiGraph, output_path: str, title: str = 'Combo Graph') -> None:
    if G.number_of_nodes() == 0:
        print('Graphe vide — pas de visualisation.')
        return

    # Filtrer les arêtes faibles (garder weight >= 2 si beaucoup d'arêtes)
    min_weight = 2 if G.number_of_edges() > 30 else 1
    G_filtered = nx.DiGraph()
    for a, b, d in G.edges(data=True):
        if d['weight'] >= min_weight:
            G_filtered.add_edge(a, b, weight=d['weight'], title=f'×{d["weight"]}')

    if G_filtered.number_of_nodes() == 0:
        G_filtered = G  # fallback : garder tout

    net = Network(height='700px', width='100%', directed=True, bgcolor='#1a1a2e', font_color='white')
    net.set_options("""{
      \"physics\": {\"enabled\": true, \"stabilization\": {\"iterations\": 200}},
      \"edges\": {\"arrows\": {\"to\": {\"enabled\": true}}, \"smooth\": {\"type\": \"curvedCW\", \"roundness\": 0.2}},
      \"nodes\": {\"font\": {\"size\": 12}}
    }""")

    # Taille des nœuds ∝ degré total
    degree = dict(G_filtered.degree())
    max_deg = max(degree.values()) if degree else 1
    in_deg_map = dict(G_filtered.in_degree())

    for node in G_filtered.nodes():
        size = 15 + 25 * (degree.get(node, 0) / max_deg)
        # Couleur selon in-degree : rouge = "destination finale", bleu = "starter"
        id_ratio = in_deg_map.get(node, 0) / (degree.get(node, 1))
        color = f'hsl({int(240 - id_ratio * 200)}, 80%, 55%)'
        net.add_node(node, label=node, size=size, color=color, title=f'Degree: {degree[node]}')

    for a, b, d in G_filtered.edges(data=True):
        width = 1 + d.get('weight', 1) * 1.5
        net.add_edge(a, b, width=width, title=d.get('title', ''))

    net.save_graph(output_path)
    print(f'Graphe sauvegardé : {output_path}')


video_id_safe = VIDEO_ID.replace('-', '_')
output_path = os.path.join(os.getcwd(), '..', 'data', f'graph_combo_{video_id_safe}.html')
visualize_combo_graph(G, output_path, title=f'Combo Graph — {VIDEO_ID}')

# Afficher dans le notebook
from IPython.display import IFrame, display
if os.path.exists(output_path):
    display(IFrame(src=output_path, width='100%', height='720px'))

Graphe sauvegardé : /Users/thomascozian/code/yugioh-meta-analyzer/notebooks/../data/graph_combo_DSYkfk5u_uA.html


## Cell 8 — Sauvegarde en DB + résumé

In [ ]:
# Sauvegarder les mentions et transitions en DB
conn = sqlite3.connect(DB)

# Créer les tables si besoin
conn.execute("""CREATE TABLE IF NOT EXISTS combo_mentions (
    video_id TEXT, card TEXT, start REAL, text TEXT
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS combo_edges (
    video_id TEXT, card_from TEXT, card_to TEXT, weight INTEGER
)""")

# Supprimer les anciennes données pour cette vidéo (idempotent)
conn.execute("DELETE FROM combo_mentions WHERE video_id = ?", (VIDEO_ID,))
conn.execute("DELETE FROM combo_edges   WHERE video_id = ?", (VIDEO_ID,))
conn.commit()

if not mentions_df.empty:
    mentions_out = mentions_df.copy()
    mentions_out['video_id'] = VIDEO_ID
    mentions_out.to_sql('combo_mentions', conn, if_exists='append', index=False)
    print(f'combo_mentions : {len(mentions_out)} lignes insérées')

if G.number_of_edges() > 0:
    edges_rows = [
        {'video_id': VIDEO_ID, 'card_from': a, 'card_to': b, 'weight': d['weight']}
        for a, b, d in G.edges(data=True)
    ]
    edges_df = pd.DataFrame(edges_rows)
    edges_df.to_sql('combo_edges', conn, if_exists='append', index=False)
    print(f'combo_edges : {len(edges_df)} arêtes insérées')

conn.commit()
conn.close()

print('\n=== RÉSUMÉ ===')
print(f'Vidéo        : {VIDEO_URL}')
print(f'Archetype    : {ARCHETYPE}')
print(f'Segments     : {len(transcript)}')
print(f'Mentions     : {len(mentions_df)} ({mentions_df["card"].nunique() if not mentions_df.empty else 0} cartes uniques)')
print(f'Combos       : {len(combos)} séquences')
print(f'Graphe       : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes')
print(f'Visualisation: {output_path}')

## Cell 9 — Étendre à plusieurs vidéos

Pour analyser plusieurs vidéos automatiquement, lister les video IDs et reboucler.

```python
VIDEO_IDS = [
    ('ct9CxAYU5ZM', 'Kewl Tune'),   # KEWL TUNE 101 — guide complet
    ('CmYgn73Nalg', 'Kewl Tune'),   # Ultimate Kewl Tune Combo Guide
    ('dkY-gFvSiEE', 'Kewl Tune'),   # Most Comprehensive Kewl Tune Guide
    # Ajouter d'autres IDs ici
]

all_mentions, all_combos = [], []

for vid, arch in VIDEO_IDS:
    t = fetch_transcript(vid)
    if not t: continue
    t_norm = [{'text': normalize_transcript(s['text'], arch), 'start': s['start'], 'duration': s['duration']} for s in t]
    m = extract_card_mentions(t_norm, card_patterns)
    m_df = pd.DataFrame(m).sort_values('start')
    c = segment_combos(m_df, COMBO_WINDOW_SEC)
    all_mentions.extend(m)
    all_combos.extend(c)
    print(f'{vid} ({arch}): {len(m)} mentions, {len(c)} combos')

G_global = build_combo_graph(all_combos)
visualize_combo_graph(G_global, '../data/graph_combo_global.html')
```

**Résultats validés — Kewl Tune (ct9CxAYU5ZM, 41min) :**
- 161 mentions, 20 cartes uniques, 24 combos, 18 nœuds, 44 arêtes
- Top starters : Mix (×35), Rotary (×29), Harmonia (×23), Clip (×17), Remix (×17)
- Combo principal : Mix → Remix → Track Maker (×4-5 chaque transition)
- ⚠️ Nécessite `ASR_CORRECTIONS` + `ARCHETYPE_NICKNAMES` : YouTube écrit "Cool Tune" (×14)

**Prochaines sources :**
- Canaux connus : NimbleSummoner, TeamSamuraiX1, Farfa, Rata
- Avec une clé YouTube Data API → `search` automatique par archetype